# Gold: service portfolio

**Audience:** data engineers validating medallion architecture and AIDP lineage.

**Prerequisites:** the canonical lab assets, shared compute and five job parameters.

**Learning goals:** trace governed transformations, verify isolation, and inspect deterministic results.


In [ ]:
import re
from functools import reduce
from pyspark.sql import Window, functions as F

# oidlUtils is injected by AIDP Workbench; no import is required.
def required_parameter(name):
    value = oidlUtils.parameters.getParameter(name, "")
    if value is None or not str(value).strip():
        raise ValueError(f"Missing AIDP job parameter: {name}")
    return str(value).strip()

participant_key = required_parameter("participant_key")
lab_id = required_parameter("lab_id")
workspace_root = required_parameter("workspace_root")
bucket_name = required_parameter("bucket_name")
objectstorage_namespace = required_parameter("objectstorage_namespace")
catalog_name = required_parameter("catalog_name")

participant_match = re.fullmatch(r"u([1-9][0-9]*)", participant_key)
if participant_match is None or int(participant_match.group(1)) < 101:
    raise ValueError("Invalid participant_key")
if lab_id != "telco_lineage":
    raise ValueError("This notebook belongs to a different lab")
if not workspace_root.startswith("/Workspace/medallon/"):
    raise ValueError("Invalid workspace_root")
if catalog_name != f"{participant_key}_aidp_lab":
    raise ValueError("Invalid participant catalog")
spark.conf.set("spark.aidp.lineage.enabled", "true")

layer_prefixes = {"landing": "01_landing", "bronze": "02_bronze", "silver": "03_silver", "gold": "04_gold"}

def table(layer, logical_name):
    return f"{catalog_name}.oci_{layer}.{participant_key}_{lab_id}_{logical_name}"

def location(layer, logical_name):
    return f"oci://{bucket_name}@{objectstorage_namespace}/{layer_prefixes[layer]}/users/{participant_key}/{lab_id}/{logical_name}/"

def write_delta(frame, layer, logical_name, _ddl):
    target = table(layer, logical_name)
    (frame.write.format("delta").mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(target))
    actual = spark.table(target).count()
    assert actual == frame.count(), f"Delta count mismatch for {logical_name}"
    print(f"Delta {layer}.{logical_name}: {actual} rows")


## Transformation

Run this cell once. It is idempotent and checks its row-level contract.


In [ ]:
ownership = spark.table(table("silver", "service_ownership"))
customers = spark.table(table("silver", "customer_master")).select("customer_id", "full_name", "document_number")
products = spark.table(table("silver", "product_catalog")).select("product_id", "product_name", "product_family")
addresses = spark.table(table("silver", "customer_addresses"))
address_window = Window.partitionBy("customer_id").orderBy(F.col("is_primary").desc(), F.col("updated_at").desc(), F.col("address_id"))
primary_address = (addresses.withColumn("_rank", F.row_number().over(address_window)).filter(F.col("_rank") == 1)
    .select("customer_id", F.col("address_id").alias("customer_address_id"), "city", "province", "region"))
customer_service_portfolio = (ownership.join(customers, "customer_id")
    .join(products, "product_id").join(primary_address, "customer_id", "left")
    .withColumn("service_address_id", F.coalesce("address_id", "customer_address_id"))
    .select("participant_key", "customer_id", "full_name", "document_number", "service_id", "service_number",
        "service_type", "product_id", "product_name", "product_family", "status", "monthly_value",
        "service_address_id", "city", "province", "region", "technology"))
write_delta(customer_service_portfolio, "gold", "customer_service_portfolio", "participant_key STRING, customer_id STRING, full_name STRING, document_number STRING, service_id STRING, service_number STRING, service_type STRING, product_id STRING, product_name STRING, product_family STRING, status STRING, monthly_value DECIMAL(14,2), service_address_id STRING, city STRING, province STRING, region STRING, technology STRING")
assert customer_service_portfolio.count() == 1261


## Exercise and common pitfall

**Exercise:** follow one customer or service identifier into the next task and explain every derived column.

**Answer scaffold:** identify the source table, join key, transformation and target column.

**Pitfall:** never replace the job parameters with participant-specific literals; doing so breaks canonical hashes and isolation.

**Extension:** inspect the resulting entity and column lineage in Master Catalog.
